## 1. Imports & Configuration

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
SALES_FILE = RAW_DIR / "retail_sales_ml_apl.csv"
INVENTORY_FILE = RAW_DIR / "retail_inventory_ml_apl.csv"

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)


Project root: /
Raw directory: /data/raw


## 2. Load Raw Data

In [5]:
SALES_FILE = Path("/content/retail_sales_ml_apl.csv")
INVENTORY_FILE = Path("/content/retail_inventory_ml_apl.csv")

sales = pd.read_csv(SALES_FILE)
inventory = pd.read_csv(INVENTORY_FILE)

print(f"Sales shape: {sales.shape}")
print(f"Inventory shape: {inventory.shape}")

Sales shape: (125751, 17)
Inventory shape: (284755, 18)


## 3. Completeness — Missing Values

In [6]:
def missing_report(df, dataset_name):
    out = pd.DataFrame({
        "Dataset": dataset_name,
        "Column": df.columns,
        "Missing_Count": df.isna().sum().values,
        "Missing_%": df.isna().mean().values * 100
    })
    return out.sort_values(["Missing_Count", "Column"], ascending=[False, True])

sales_missing = missing_report(sales, "Sales")
inventory_missing = missing_report(inventory, "Inventory")

display(sales_missing)
display(inventory_missing)


,Dataset,Column,Missing_Count,Missing_%
3,Sales,Reason of Return,125751,100.0
15,Sales,Cogs,0,0.0
2,Sales,Is Return,0,0.0
16,Sales,Number of Transactions,0,0.0
8,Sales,Product Category,0,0.0
6,Sales,Product Description,0,0.0
7,Sales,Product Division,0,0.0
5,Sales,Product No,0,0.0
10,Sales,Product Segment,0,0.0
9,Sales,Product Subcategory,0,0.0


,Dataset,Column,Missing_Count,Missing_%
15,Inventory,Cost of Stocks,0,0.0
1,Inventory,End Date,0,0.0
7,Inventory,Product Category,0,0.0
5,Inventory,Product Description,0,0.0
6,Inventory,Product Division,0,0.0
4,Inventory,Product No,0,0.0
9,Inventory,Product Segment,0,0.0
8,Inventory,Product Subcategory,0,0.0
13,Inventory,Qty on hand,0,0.0
12,Inventory,Sales Channel,0,0.0


## 4. Uniqueness — Exact Duplicate Records

In [7]:
duplicate_summary = pd.DataFrame({
    "Dataset": ["Sales", "Inventory"],
    "Exact_Duplicates": [sales.duplicated().sum(), inventory.duplicated().sum()],
    "Duplicate_%": [sales.duplicated().mean()*100, inventory.duplicated().mean()*100]
})
display(duplicate_summary)


,Dataset,Exact_Duplicates,Duplicate_%
0,Sales,0,0.0
1,Inventory,0,0.0


## 5. Business-Key Duplicate Analysis — Sales

In [10]:
sales["Date_parsed"] = pd.to_datetime(sales["Transaction Date"], errors="coerce")

sales_key_counts = (
    sales.groupby(["Store", "Product No", "Date_parsed"], dropna=False)
         .size()
         .reset_index(name="Record_Count")
)

sales_repeated_keys = sales_key_counts[sales_key_counts["Record_Count"] > 1].copy()

print(f"Repeated Store × Product × Date keys: {len(sales_repeated_keys):,}")
display(sales_repeated_keys.head(20))

Repeated Store × Product × Date keys: 1,823


,Store,Product No,Date_parsed,Record_Count
42,STR-1006,PROD-100360,2025-06-18,2
58,STR-1006,PROD-100433,2025-09-04,2
222,STR-1006,PROD-100453,2026-03-21,2
242,STR-1006,PROD-100459,2025-10-03,2
416,STR-1006,PROD-100805,2025-10-28,2
424,STR-1006,PROD-100805,2025-12-20,2
432,STR-1006,PROD-100805,2026-02-12,2
433,STR-1006,PROD-100805,2026-02-13,2
634,STR-1006,PROD-115348,2025-12-24,2
651,STR-1006,PROD-116357,2025-12-03,2


In [14]:
if len(sales_repeated_keys):
    repeated_examples = sales.merge(
        sales_repeated_keys[["Store", "Product No", "Date_parsed"]],
        on=["Store", "Product No", "Date_parsed"],
        how="inner"
    ).sort_values(["Store", "Product No", "Date_parsed"])
    display(repeated_examples.head(30))

,Transaction Date,Sales Type,Is Return,Reason of Return,Supplier,Product No,Product Description,Product Division,Product Category,Product Subcategory,Product Segment,Store,Sales Channel,Qty Sold,Sales Amount,Cogs,Number of Transactions,Date_parsed
527,2025-06-18,Promo,1,NaN,Vendor 0166,PROD-100360,PROD-164482 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr,STR-1006,Channel Alpha,-1.0,-89.99,-59.26,1,2025-06-18
1350,2025-06-18,Promo,0,NaN,Vendor 0166,PROD-100360,PROD-164482 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr,STR-1006,Channel Alpha,1.0,89.99,59.26,1,2025-06-18
1463,2025-09-04,Promo,0,NaN,Vendor 0166,PROD-100433,PROD-164663 SEG-515333 Scholar Footwear Court ...,Scholar Footwear,Scholar Footwear Court Casual,Scholar Footwear Court Casual Zephyr,SEG-515333 Scholar Footwear Court Casual Zephyr,STR-1006,Channel Alpha,1.0,69.99,46.51,1,2025-09-04
2967,2025-09-04,Promo,1,NaN,Vendor 0166,PROD-100433,PROD-164663 SEG-515333 Scholar Footwear Court ...,Scholar Footwear,Scholar Footwear Court Casual,Scholar Footwear Court Casual Zephyr,SEG-515333 Scholar Footwear Court Casual Zephyr,STR-1006,Channel Alpha,-1.0,-69.99,-46.51,1,2025-09-04
922,2026-03-21,Full Price,0,NaN,Vendor 0166,PROD-100453,PROD-100829 SEG-500112 Heritage Footwear Fligh...,Femme Footwear,Femme Footwear Flight Classic,Femme Footwear Flight Classic Varied,SEG-500112 Heritage Footwear Flight Classic Ze...,STR-1006,Channel Alpha,1.0,115.00,58.15,1,2026-03-21
2396,2026-03-21,Full Price,1,NaN,Vendor 0166,PROD-100453,PROD-100829 SEG-500112 Heritage Footwear Fligh...,Femme Footwear,Femme Footwear Flight Classic,Femme Footwear Flight Classic Varied,SEG-500112 Heritage Footwear Flight Classic Ze...,STR-1006,Channel Alpha,-1.0,-115.00,-58.15,1,2026-03-21
3072,2025-10-03,Full Price,1,NaN,Vendor 0166,PROD-100459,PROD-100823 SEG-500112 Heritage Footwear Fligh...,Femme Footwear,Femme Footwear Flight Classic,Femme Footwear Flight Classic Varied,SEG-500112 Heritage Footwear Flight Classic Ze...,STR-1006,Channel Alpha,-1.0,-115.00,-58.12,1,2025-10-03
3211,2025-10-03,Full Price,0,NaN,Vendor 0166,PROD-100459,PROD-100823 SEG-500112 Heritage Footwear Fligh...,Femme Footwear,Femme Footwear Flight Classic,Femme Footwear Flight Classic Varied,SEG-500112 Heritage Footwear Flight Classic Ze...,STR-1006,Channel Alpha,1.0,115.00,58.12,1,2025-10-03
2225,2025-10-28,Full Price,0,NaN,Vendor 0166,PROD-100805,PROD-102578 SEG-500351 Scholar Footwear Flight...,Scholar Footwear,Scholar Footwear Flight Classic,Scholar Footwear Flight Classic Zephyr,SEG-500351 Scholar Footwear Flight Classic Zephyr,STR-1006,Channel Alpha,1.0,90.00,45.58,1,2025-10-28
2742,2025-10-28,Full Price,1,NaN,Vendor 0166,PROD-100805,PROD-102578 SEG-500351 Scholar Footwear Flight...,Scholar Footwear,Scholar Footwear Flight Classic,Scholar Footwear Flight Classic Zephyr,SEG-500351 Scholar Footwear Flight Classic Zephyr,STR-1006,Channel Alpha,-1.0,-90.00,-45.58,1,2025-10-28


## 6. Validity — Sales Quantity & Returns

In [12]:
sales_qty = sales["Qty Sold"]

checks = pd.Series({
    "Negative Qty Sold": (sales_qty < 0).sum(),
    "Zero Qty Sold": (sales_qty == 0).sum(),
    "Positive Qty Sold": (sales_qty > 0).sum(),
    "Missing Qty Sold": sales_qty.isna().sum()
})
display(checks.to_frame("Count"))

negative_sales = sales[sales["Qty Sold"] < 0].copy()
print(f"Negative-quantity records: {len(negative_sales):,}")
display(negative_sales.head(20))


,Count
Negative Qty Sold,7491
Zero Qty Sold,2
Positive Qty Sold,118258
Missing Qty Sold,0


Negative-quantity records: 7,491


,Transaction Date,Sales Type,Is Return,Reason of Return,Supplier,Product No,Product Description,Product Division,Product Category,Product Subcategory,Product Segment,Store,Sales Channel,Qty Sold,Sales Amount,Cogs,Number of Transactions,Date_parsed
39,2026-03-14,Full Price,1,NaN,Vendor 0134,PROD-183451,PROD-114074 SEG-502516 Heritage Footwear Retro...,Scholar Footwear,Scholar Footwear Retro Legend,Scholar Footwear Retro Legend Summit,SEG-502516 Heritage Footwear Retro Legend Summit,STR-1109,Channel Alpha,-1.0,-165.00,-83.49,1,2026-03-14
47,2025-07-30,Promo,1,NaN,Vendor 0166,PROD-144341,PROD-152446 SEG-512298 Scholar Footwear Court ...,Scholar Footwear,Scholar Footwear Court Casual,Scholar Footwear Court Casual Zephyr,SEG-512298 Scholar Footwear Court Casual Zephyr,STR-1369,Channel Alpha,-1.0,-69.99,-50.05,1,2025-07-30
68,2026-03-14,Full Price,1,NaN,Vendor 0165,PROD-188440,PROD-122939 SEG-504714 Junior Apparel Cozy Tro...,Junior Apparel,Junior Apparel Cozy Trousers,Junior Apparel Cozy Trousers Varied,SEG-504714 Junior Apparel Cozy Trousers Varied,STR-1010,Channel Alpha,-1.0,-60.00,-23.40,1,2026-03-14
115,2025-12-29,Full Price,1,NaN,Vendor 0169,PROD-152224,PROD-149105 SEG-511422 Scholar Footwear Trailb...,Scholar Footwear,Scholar Footwear Trailblazer,Scholar Footwear Trailblazer Equilibrium,SEG-511422 Scholar Footwear Trailblazer Equili...,STR-1006,Channel Alpha,-1.0,-63.00,-38.67,1,2025-12-29
151,2025-08-23,Full Price,1,NaN,Vendor 0134,PROD-160814,PROD-166159 SEG-515580 Scholar Footwear Modern...,Scholar Footwear,Scholar Footwear Modern Legend,Scholar Footwear Modern Legend Summit,SEG-515580 Scholar Footwear Modern Legend Summit,STR-1105,Channel Alpha,-1.0,-120.00,-60.06,1,2025-08-23
153,2025-08-28,Full Price,1,NaN,Vendor 0169,PROD-143184,PROD-102483 SEG-500343 Scholar Footwear Trailb...,Scholar Footwear,Scholar Footwear Trailblazer,Scholar Footwear Trailblazer Equilibrium,SEG-500343 Scholar Footwear Trailblazer Equili...,STR-1338,Channel Alpha,-1.0,-110.00,-50.05,1,2025-08-28
162,2026-01-02,Full Price,1,NaN,Vendor 0166,PROD-100813,PROD-100959 SEG-500123 Scholar Footwear Flight...,Scholar Footwear,Scholar Footwear Flight Classic,Scholar Footwear Flight Classic Zephyr,SEG-500123 Scholar Footwear Flight Classic Zephyr,STR-1348,Channel Alpha,-1.0,-90.00,-45.51,1,2026-01-02
168,2025-12-24,Full Price,1,NaN,Vendor 0064,PROD-100043,PROD-189472 SEG-520541 Femme Footwear Boot Col...,Femme Footwear,Femme Footwear Boot Collection,Femme Footwear Boot Collection Plushfoot,SEG-520541 Femme Footwear Boot Collection Plus...,STR-1074,Channel Alpha,-1.0,-112.00,-67.99,1,2025-12-24
190,2025-09-14,Full Price,1,NaN,Vendor 0134,PROD-167815,PROD-114497 SEG-502552 Scholar Footwear Modern...,Scholar Footwear,Scholar Footwear Modern Legend,Scholar Footwear Modern Legend Summit,SEG-502552 Scholar Footwear Modern Legend Summit,STR-1344,Channel Alpha,-1.0,-110.00,-55.66,1,2025-09-14
200,2025-12-10,Full Price,1,NaN,Vendor 0166,PROD-176331,PROD-111478 SEG-502269 Scholar Footwear Flight...,Scholar Footwear,Scholar Footwear Flight Classic,Scholar Footwear Flight Classic Zephyr,SEG-502269 Scholar Footwear Flight Classic Zephyr,STR-1305,Channel Alpha,-1.0,-81.00,-45.54,1,2025-12-10


## 7. Validity — Inventory Quantity

In [13]:
inventory_qty = inventory["Qty on hand"]

inventory_checks = pd.Series({
    "Negative Qty on hand": (inventory_qty < 0).sum(),
    "Zero Qty on hand": (inventory_qty == 0).sum(),
    "Positive Qty on hand": (inventory_qty > 0).sum(),
    "Missing Qty on hand": inventory_qty.isna().sum()
})
display(inventory_checks.to_frame("Count"))

negative_inventory = inventory[inventory["Qty on hand"] < 0].copy()
print(f"Negative inventory records: {len(negative_inventory):,}")
display(negative_inventory.head(30))


,Count
Negative Qty on hand,1380
Zero Qty on hand,13869
Positive Qty on hand,269506
Missing Qty on hand,0


Negative inventory records: 1,380


,Start Date,End Date,Stock Status,Supplier,Product No,Product Description,Product Division,Product Category,Product Subcategory,Product Segment,Store,Store Type,Sales Channel,Qty on hand,Stocks Selling Amount,Cost of Stocks,Stock Unit Selling Price,Stock Unit Cost Price
11,2025-09-06,2026-01-05,Full Price,Vendor 0166,PROD-100353,PROD-164489 SEG-515310 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-515310 Femme Footwear Court Casual Zephyr,STR-1145,Retail Storefront,Channel Alpha,-1,-120.00,-59.17,120.00,59.1700
298,2025-10-28,2026-01-04,Markdown Tier 2,Vendor 0198,PROD-148307,PROD-117613 SEG-503080 Junior Apparel Printed ...,Junior Apparel,Junior Apparel Printed Tees,Junior Apparel Printed Tees Varied,SEG-503080 Junior Apparel Printed Tees Varied,STR-1157,Retail Storefront,Channel Alpha,-2,-9.98,-14.50,4.99,7.2500
463,2025-10-26,2025-10-26,Markdown Tier 2,Vendor 0134,PROD-139506,PROD-118198 SEG-503268 Scholar Footwear Retro ...,Scholar Footwear,Scholar Footwear Retro Legend,Scholar Footwear Retro Legend Summit,SEG-503268 Scholar Footwear Retro Legend Summit,STR-1163,Retail Storefront,Channel Alpha,-1,-39.99,-70.07,39.99,70.0700
1059,2026-03-14,9999-12-31,Markdown Tier 2,Vendor 0166,PROD-161861,PROD-170262 SEG-516305 Junior Apparel Trousers...,Junior Apparel,Junior Apparel Trousers,Junior Apparel Trousers Varied,SEG-516305 Junior Apparel Trousers Varied,STR-1107,Retail Storefront,Channel Alpha,-1,-14.99,-15.92,14.99,15.9200
1372,2026-03-04,2026-03-04,Full Price,Vendor 0166,PROD-175050,PROD-166383 SEG-515621 Scholar Footwear Trailb...,Scholar Footwear,Scholar Footwear Trailblazer,Scholar Footwear Trailblazer Zephyr,SEG-515621 Scholar Footwear Trailblazer Zephyr,STR-1222,Retail Storefront,Channel Alpha,-1,-147.00,-74.38,147.00,74.3800
1769,2026-04-05,2026-04-09,Markdown Tier 1,Vendor 0010,PROD-153852,PROD-134075 SEG-507544 Junior Apparel Miscella...,Junior Apparel,Junior Apparel Miscellany,Junior Apparel Miscellany Varied,SEG-507544 Junior Apparel Miscellany Varied,STR-1198,Retail Storefront,Channel Alpha,-1,-24.99,-23.50,24.99,23.5000
2203,2025-12-19,2026-01-05,Full Price,Vendor 0262,PROD-163861,PROD-180137 SEG-518154 Femme Footwear Trailbla...,Femme Footwear,Femme Footwear Trailblazer,Femme Footwear Trailblazer Swift Run,SEG-518154 Femme Footwear Trailblazer Swift Run,STR-1010,Retail Storefront,Channel Alpha,-2,-200.00,-99.16,100.00,49.5800
2246,2025-06-25,2025-12-07,Full Price,Vendor 0166,PROD-152547,PROD-128745 SEG-506280 Femme Footwear Court Ca...,Femme Footwear,Femme Footwear Court Casual,Femme Footwear Court Casual Zephyr,SEG-506280 Femme Footwear Court Casual Zephyr,STR-1044,Retail Storefront,Channel Alpha,-1,-135.00,-67.58,135.00,67.5800
2708,2026-01-05,2026-01-06,Full Price,Vendor 0166,PROD-124596,PROD-100991 SEG-500125 Toddler Footwear Flight...,Scholar Footwear,Scholar Footwear Flight Classic,Scholar Footwear Flight Classic Zephyr,SEG-500125 Toddler Footwear Flight Classic Zephyr,STR-1038,Retail Storefront,Channel Alpha,-1,-90.00,-40.45,90.00,40.4500
2711,2025-11-14,2026-01-06,Full Price,Vendor 0134,PROD-152045,PROD-166020 SEG-515561 Scholar Footwear Modern...,Scholar Footwear,Scholar Footwear Modern Legend,Scholar Footwear Modern Legend Summit,SEG-515561 Scholar Footwear Modern Legend Summit,STR-1044,Retail Storefront,Channel Alpha,-1,-110.00,-55.09,110.00,55.0900


## 8. Temporal Integrity — Sales Dates

In [15]:
sales_dates = sales["Date_parsed"].dropna().sort_values().drop_duplicates()
sales_min = sales_dates.min()
sales_max = sales_dates.max()

expected_dates = pd.date_range(sales_min, sales_max, freq="D")
missing_sales_dates = expected_dates.difference(sales_dates)

print("Sales start:", sales_min.date())
print("Sales end  :", sales_max.date())
print("Distinct sales dates:", len(sales_dates))
print("Missing calendar dates:", len(missing_sales_dates))
display(pd.DataFrame({"Missing_Sales_Date": missing_sales_dates}))


Sales start: 2025-06-01
Sales end  : 2026-04-24
Distinct sales dates: 326
Missing calendar dates: 2


,Missing_Sales_Date
0,2025-11-27
1,2025-12-25


## 9. Temporal Integrity — Inventory Intervals

In [16]:
inventory["Start_Date_parsed"] = pd.to_datetime(inventory["Start Date"], errors="coerce")
inventory["End_Date_parsed"] = pd.to_datetime(inventory["End Date"], errors="coerce")

sentinel = pd.Timestamp("9999-12-31")
sentinel_mask = inventory["End_Date_parsed"] == sentinel

print(f"Open-ended/sentinel intervals: {sentinel_mask.sum():,}")
print(f"Equal start/end dates: {(inventory['Start_Date_parsed'] == inventory['End_Date_parsed']).sum():,}")
print(f"End before start: {(inventory['End_Date_parsed'] < inventory['Start_Date_parsed']).sum():,}")


Open-ended/sentinel intervals: 0
Equal start/end dates: 34,998
End before start: 0


In [19]:
intervals = inventory[["Store","Product No","Start_Date_parsed","End_Date_parsed"]].copy()
intervals["End_For_Gap"] = intervals["End_Date_parsed"].where(~sentinel_mask, pd.NaT)
intervals = intervals.sort_values(["Store","Product No","Start_Date_parsed"])
intervals["Previous_End"] = intervals.groupby(["Store","Product No"])["End_For_Gap"].shift(1)
intervals["Gap_Days"] = (intervals["Start_Date_parsed"] - intervals["Previous_End"]).dt.days - 1

gaps = intervals[intervals["Gap_Days"] > 0]
print(f"Inventory interval starts following a gap: {len(gaps):,}")
display(gaps[["Store","Product No","Previous_End","Start_Date_parsed","Gap_Days"]].head(30))

Inventory interval starts following a gap: 8,607


,Store,Product No,Previous_End,Start_Date_parsed,Gap_Days
144773,STR-1006,PROD-100043,2025-12-23,2026-03-17,83.0
75082,STR-1006,PROD-100044,2026-02-05,2026-03-17,39.0
22487,STR-1006,PROD-100045,2026-01-31,2026-03-17,44.0
249519,STR-1006,PROD-100433,2026-01-04,2026-01-23,18.0
94469,STR-1006,PROD-100434,2026-01-04,2026-01-23,18.0
94470,STR-1006,PROD-100435,2026-01-15,2026-01-23,7.0
281816,STR-1006,PROD-103734,2025-10-16,2025-12-18,62.0
196395,STR-1006,PROD-103735,2025-10-16,2025-12-16,60.0
54827,STR-1006,PROD-103737,2025-10-11,2025-12-12,61.0
113036,STR-1006,PROD-110041,2025-11-14,2025-11-24,9.0


## 10. Sales ↔ Inventory Referential Coverage

In [24]:
sales_keys = sales[["Store","Product No"]].drop_duplicates()
inventory_keys = inventory[["Store","Product No"]].drop_duplicates()

coverage = sales_keys.merge(
    inventory_keys,
    on=["Store","Product No"],
    how="outer",
    indicator=True
)

coverage_summary = (
    coverage["_merge"]
    .value_counts()
    .rename_axis("Coverage")
    .reset_index(name="Store_Product_Combinations")
)
coverage_summary["Share_%"] = coverage_summary["Store_Product_Combinations"] / len(coverage) * 100
display(coverage_summary)

print("Products in Sales only:", len(set(sales["Product No"]) - set(inventory["Product No"])))
print("Products in Inventory only:", len(set(inventory["Product No"]) - set(sales["Product No"])))


,Coverage,Store_Product_Combinations,Share_%
0,both,47859,75.302096
1,right_only,13109,20.625905
2,left_only,2588,4.071999


Products in Sales only: 0
Products in Inventory only: 0


## 11. Product Attribute Consistency

In [21]:
candidate_columns = [
    "Product Description", "Division", "Category",
    "Subcategory", "Segment", "Supplier"
]
candidate_columns = [c for c in candidate_columns if c in sales.columns and c in inventory.columns]

results = []
for col in candidate_columns:
    s = sales.groupby("Product No")[col].nunique(dropna=False)
    i = inventory.groupby("Product No")[col].nunique(dropna=False)
    results.append({
        "Attribute": col,
        "Products_with_multiple_sales_values": int((s > 1).sum()),
        "Products_with_multiple_inventory_values": int((i > 1).sum())
    })

display(pd.DataFrame(results))


,Attribute,Products_with_multiple_sales_values,Products_with_multiple_inventory_values
0,Product Description,0,0
1,Supplier,0,0


## 12. Financial Consistency — Inventory Values

In [25]:
inventory["Expected_Stock_Selling_Amount"] = (
    inventory["Qty on hand"] * inventory["Stock Unit Selling Price"]
)
inventory["Selling_Amount_Difference"] = (
    inventory["Stocks Selling Amount"] - inventory["Expected_Stock_Selling_Amount"]
)

selling_fail = inventory["Selling_Amount_Difference"].abs() > 0.01
print(f"Selling amount records outside £0.01 tolerance: {selling_fail.sum():,}")
display(inventory.loc[selling_fail, [
    "Qty on hand", "Stock Unit Selling Price",
    "Stocks Selling Amount", "Expected_Stock_Selling_Amount",
    "Selling_Amount_Difference"
]].head(20))


Selling amount records outside £0.01 tolerance: 0


,Qty on hand,Stock Unit Selling Price,Stocks Selling Amount,Expected_Stock_Selling_Amount,Selling_Amount_Difference


In [26]:
inventory["Expected_Cost_of_Stocks"] = (
    inventory["Qty on hand"] * inventory["Stock Unit Cost Price"]
)
inventory["Cost_Amount_Difference"] = (
    inventory["Cost of Stocks"] - inventory["Expected_Cost_of_Stocks"]
)

cost_fail = inventory["Cost_Amount_Difference"].abs() > 0.01
print(f"Cost-of-stocks records outside £0.01 tolerance: {cost_fail.sum():,}")
display(inventory.loc[cost_fail, [
    "Qty on hand", "Stock Unit Cost Price",
    "Cost of Stocks", "Expected_Cost_of_Stocks",
    "Cost_Amount_Difference"
]].head(20))


Cost-of-stocks records outside £0.01 tolerance: 8,402


,Qty on hand,Stock Unit Cost Price,Cost of Stocks,Expected_Cost_of_Stocks,Cost_Amount_Difference
5,0,0.0,-6.80,0.0,-6.80
12,0,0.0,5.46,0.0,5.46
13,0,0.0,-0.15,0.0,-0.15
14,0,0.0,-1.63,0.0,-1.63
69,0,0.0,-0.06,0.0,-0.06
143,0,0.0,-0.03,0.0,-0.03
145,0,0.0,0.11,0.0,0.11
148,0,0.0,0.17,0.0,0.17
207,0,0.0,-1.40,0.0,-1.40
236,0,0.0,3.74,0.0,3.74


## 13. Forecasting Readiness

In [27]:
readiness = pd.Series({
    "Date available": "Date" in sales.columns,
    "Store available": "Store" in sales.columns,
    "Product available": "Product" in sales.columns,
    "Demand available": "Qty Sold" in sales.columns,
    "Observed inventory available": "Qty on hand" in inventory.columns,
    "Product hierarchy available": any(c in sales.columns for c in ["Category","Subcategory","Segment"]),
    "Price/revenue information available": any(c in sales.columns for c in ["Sales Amount","Selling Price","Unit Selling Price"])
}, name="Available")

display(readiness.to_frame())


,Available
Date available,False
Store available,True
Product available,False
Demand available,True
Observed inventory available,True
Product hierarchy available,False
Price/revenue information available,True


## 14. Inventory Engine Readiness

In [28]:
inventory_engine = pd.DataFrame([
    ["Historical demand", "AVAILABLE", "Use for demand modelling"],
    ["Observed inventory quantity", "AVAILABLE", "Use after anomaly treatment"],
    ["Store", "AVAILABLE", "Dimension"],
    ["Product", "AVAILABLE", "Dimension"],
    ["Supplier", "AVAILABLE", "Dimension"],
    ["Price / cost", "AVAILABLE", "Business-value analysis"],
    ["Historical purchase orders", "NOT OBSERVED", "Do not fabricate"],
    ["Historical lead time", "NOT OBSERVED", "Use documented scenario assumptions initially"],
    ["In-transit quantity", "NOT OBSERVED", "Do not fabricate"],
], columns=["Component","Status","Treatment"])

display(inventory_engine)


,Component,Status,Treatment
0,Historical demand,AVAILABLE,Use for demand modelling
1,Observed inventory quantity,AVAILABLE,Use after anomaly treatment
2,Store,AVAILABLE,Dimension
3,Product,AVAILABLE,Dimension
4,Supplier,AVAILABLE,Dimension
5,Price / cost,AVAILABLE,Business-value analysis
6,Historical purchase orders,NOT OBSERVED,Do not fabricate
7,Historical lead time,NOT OBSERVED,Use documented scenario assumptions initially
8,In-transit quantity,NOT OBSERVED,Do not fabricate


## 15. Final Quality Scorecard

In [29]:
scorecard = pd.DataFrame([
    ["Completeness", "Missing values reviewed", "PASS / REVIEW"],
    ["Uniqueness", "Exact duplicates checked", "PASS"],
    ["Business-key uniqueness", "Store × Product × Date reviewed", "REVIEW"],
    ["Sales validity", "Negative quantities linked to return logic", "REVIEW"],
    ["Inventory validity", "Negative on-hand quantities identified", "REVIEW"],
    ["Temporal integrity", "Sales dates and inventory intervals checked", "REVIEW"],
    ["Referential coverage", "Sales ↔ inventory overlap quantified", "PASS WITH LIMITATION"],
    ["Attribute consistency", "Product attributes compared", "PASS / REVIEW"],
    ["Financial consistency", "Inventory value formulas checked", "REVIEW"],
    ["Forecasting readiness", "Demand/store/product/date available", "PASS"],
    ["Inventory readiness", "Observed on-hand inventory available", "PASS WITH LIMITATION"],
], columns=["Dimension","Test","Initial Status"])

display(scorecard)


,Dimension,Test,Initial Status
0,Completeness,Missing values reviewed,PASS / REVIEW
1,Uniqueness,Exact duplicates checked,PASS
2,Business-key uniqueness,Store × Product × Date reviewed,REVIEW
3,Sales validity,Negative quantities linked to return logic,REVIEW
4,Inventory validity,Negative on-hand quantities identified,REVIEW
5,Temporal integrity,Sales dates and inventory intervals checked,REVIEW
6,Referential coverage,Sales ↔ inventory overlap quantified,PASS WITH LIMITATION
7,Attribute consistency,Product attributes compared,PASS / REVIEW
8,Financial consistency,Inventory value formulas checked,REVIEW
9,Forecasting readiness,Demand/store/product/date available,PASS
